In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from sage.all import *

# 1. Load Library
load("modular_isogeny_lib.sage")

# ==============================================================================
# Main Execution Logic
# ==============================================================================

def run_isogeny_construction(N, prec=200):
    clear_output()
    print(f"Starting computation for N = {N} with precision {prec}...")
    
    try:
        model_data = get_model_data(N)
    except NotImplementedError as e:
        print(f"\n[Error] {e}")
        return

    print("Step 1: Computing Modular Forms...")
    forms = setup_modular_forms(N, prec)
    
    print("Step 2: Retrieving X0(N) Model Generators...")
    try:
        x0, y0 = model_data['generators'](forms['q'])
    except NotImplementedError as e:
        print(f"\n[Error] {e}")
        return
    
    print("Step 3: Solving for Rational Maps (Adaptive Degree)...")
    P_xy = PolynomialRing(QQ, names=['x', 'y'])
    
    # --------------------------------------------------------------------------
    # Helper: Adaptive Search Function
    # --------------------------------------------------------------------------
    def find_map_adaptive(target_f, label, start_deg=10, max_deg=100, step=5):
        """
        Tries to find the rational map. If failed, increases degree and retries.
        Prints a message if not found at the current degree step.
        """
        if N == 67:
            start_deg = 20  # Start higher for N=67 due to complexity
        if N == 163:
            start_deg = 45  # Start higher for N=163 due to complexity
        for deg in range(start_deg, max_deg + 1, step):
            # Display progress
            print(f"  - Trying {label} with max_degree={deg}...", end="\r")
            
            sols = find_minimal_robust_solution(target_f, x0, y0, P_xy, 
                                                max_total_degree=deg, 
                                                max_deg_y=deg//3,
                                                verbose=False)
            if sols:
                print(f"  -> Found {label} (max_degree={deg})")
                return AB_ratio(sols[0])
            else:
                # Print message if not found at current degree
                print(f"  -> Could not find {label} within degree {deg}.")
    
        print(f"  -> Failed to find {label} even at max_degree={max_deg}")
        return None

    # --------------------------------------------------------------------------
    # Execute Search
    # --------------------------------------------------------------------------
    u = find_map_adaptive(forms['f'], "u")
    v = find_map_adaptive(forms['g'], "v")
    uN = find_map_adaptive(forms['fN'], "u_N")
    vN = find_map_adaptive(forms['gN'], "v_N")
    if not all([u, v, uN, vN]):
        print("\n[Warning] Failed to find some rational maps. Try increasing precision or max degree limit.")
        return

    print("\n" + "="*60)
    print(f"RESULTS FOR N={N}")
    print("="*60)
    generate_latex_formulas(u, v, uN, vN, model_data['poly_xy'])
    print("\n" + "="*60)
    # Pass the entire model_data dictionary to handle both Elliptic and General cases
    analyze_isogenies(u, v, uN, vN, N, model_data)
    print("="*60 + "\n")

# ==============================================================================
# UI Widget Setup
# ==============================================================================

n_selector = widgets.Dropdown(
    options=[11, 14, 15, 17,19,21,27,37,43,67],
    value=11,
    description='Select N:',
    disabled=False,
)

run_button = widgets.Button(
    description='Run Computation',
    button_style='success',
    tooltip='Click to start calculation',
    icon='check'
)

out = widgets.Output()

def on_button_clicked(b):
    with out:
        clear_output()
        try:
            run_isogeny_construction(n_selector.value)
        except Exception as e:
            print(f"An error occurred: {e}")
            import traceback
            traceback.print_exc()

run_button._click_handlers.callbacks = [] 
run_button.on_click(on_button_clicked)

print("Select N and click Run:")
display(n_selector, run_button, out)

Select N and click Run:


Dropdown(description='Select N:', options=(11, 14, 15, 17, 19, 21, 27, 37, 43, 67), value=11)

Button(button_style='success', description='Run Computation', icon='check', style=ButtonStyle(), tooltip='Click to start calculation')

Output()